# Trích xuất lại đặc trưng từ video gốc

Đọc `data/raw/<lớp>/<id>.mp4`, đưa **từng frame qua `frame_padding` thành 1280×1280**, rồi chạy MediaPipe builtin và `src.features.extract.extract`. Kết quả thay vào `data/processed/<lớp>/<id>/` theo đúng định dạng `DataHandler`.

Video trong `data/raw` đã chứa chiều lật gương lúc thu thập nên không lật thêm. Metadata `session`, `mirror` và thời điểm tạo mẫu được giữ lại; thống kê chất lượng được tính lại. Thư mục đặc trưng cũ được chuyển vào `data/processed/.reextract_backup_<lần chạy>/`. Mẫu lỗi được báo và giữ nguyên. Sau khi chạy xong cần huấn luyện lại model bằng `train.ipynb`.


In [1]:
import sys
import importlib
from collections import Counter
from datetime import datetime
from pathlib import Path
from uuid import uuid4

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Hãy chạy Jupyter từ thư mục dự án hoặc notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

feature_module = importlib.reload(importlib.import_module('src.features.extract'))
importlib.reload(importlib.import_module('src.features.schema'))  # schema giữ FRAME_FEATURE_DIM cũ nếu không reload
importlib.reload(importlib.import_module('src.data.data_reader'))
reextract_module = importlib.reload(importlib.import_module('src.data.reextract'))
from src.data.data_reader import DataHandler
find_raw_samples = reextract_module.find_raw_samples
reextract_sample = reextract_module.reextract_sample
FEATURE_DIM = feature_module.FEATURE_DIM
FEATURE_VERSION = feature_module.FEATURE_VERSION
process_sequence = feature_module.process_sequence
from src.features.frame import FRAME_SIZE

DATA_ROOT = PROJECT_ROOT / 'data'
RAW_ROOT = DATA_ROOT / 'raw'
PROCESSED_ROOT = DATA_ROOT / 'processed'
CLASSES = None       # None = mọi lớp; hoặc ['xin chào', 'cảm ơn']
MAX_SAMPLES = None   # None = mọi video; đặt 1 để thử một mẫu trước

if FEATURE_DIM != DataHandler.FEATURE_DIM:
    raise ValueError(f'Extractor {FEATURE_DIM}D khác DataHandler {DataHandler.FEATURE_DIM}D')
print(f'Nguồn: {RAW_ROOT}')
print(f'Đích: {PROCESSED_ROOT} | frame: {FRAME_SIZE}×{FRAME_SIZE} | feature v{FEATURE_VERSION}')

Nguồn: /Users/trungtran/Desktop/vsl-translator-new/data/raw
Đích: /Users/trungtran/Desktop/vsl-translator-new/data/processed | frame: 720×720 | feature v2


## Kiểm tra danh sách video


In [2]:
videos = find_raw_samples(RAW_ROOT, CLASSES)
if MAX_SAMPLES is not None:
    videos = videos[:MAX_SAMPLES]
if not videos:
    raise RuntimeError('Không thấy video .mp4 có ID số trong data/raw/<lớp>/.')
counts = Counter(video.parent.name for video in videos)
existing = sum((PROCESSED_ROOT / video.parent.name / video.stem).is_dir() for video in videos)
print(f'{len(videos)} video | {existing} mẫu đã có đặc trưng, {len(videos) - existing} mẫu chưa có')
for name, count in sorted(counts.items()):
    print(f'  {name}: {count}')
print('Ví dụ:', videos[0].relative_to(RAW_ROOT))

45 video | 45 mẫu đã có đặc trưng, 0 mẫu chưa có
  cảm ơn: 15
  ten_la: 15
  xin chào: 15
Ví dụ: cảm ơn/0.mp4


## Trích xuất lại

Chạy ô này để xử lý danh sách ở trên. Mỗi lần chạy tạo một thư mục backup riêng. Video và checkpoint hiện có không bị sửa.


In [3]:
run_id = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:6]
backup_root = PROCESSED_ROOT / f'.reextract_backup_{run_id}'
succeeded, failed = [], []
print(f'Backup: {backup_root}')
for index, video in enumerate(videos, 1):
    try:
        result = reextract_sample(video, RAW_ROOT, PROCESSED_ROOT, backup_root)
        succeeded.append(result)
        print(f'[{index}/{len(videos)}] OK  {result["class"]}/{result["sample_id"]}: '
              f'{result["frames"]} frame, pose {result["pose_coverage"]:.0%}, '
              f'tay T {result["left_hand"]:.0%}, tay P {result["right_hand"]:.0%}', flush=True)
    except Exception as exc:
        failed.append((str(video.relative_to(RAW_ROOT)), str(exc)))
        print(f'[{index}/{len(videos)}] LỖI {video.relative_to(RAW_ROOT)}: {exc}', flush=True)
print(f'Hoàn tất: {len(succeeded)} thành công, {len(failed)} lỗi')
if failed:
    print('Mẫu lỗi vẫn giữ đặc trưng cũ; có thể sửa video rồi chạy lại.')

Backup: /Users/trungtran/Desktop/vsl-translator-new/data/processed/.reextract_backup_20261116_222103_8c41e3


I0000 00:00:1794842463.589331 11582425 init-domain.cc:134] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1794842463.665020 11582428 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842463.688734 11582428 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842463.706066 11582438 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842463.718445 11582438 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842463.763953 11582429 landmark_projection_calculator.cc:

[1/45] OK  cảm ơn/0: 19 frame, pose 100%, tay T 9%, tay P 85%


W0000 00:00:1794842464.825516 11582482 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842464.832595 11582487 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842464.837705 11582514 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842464.841175 11582514 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[2/45] OK  cảm ơn/1: 21 frame, pose 100%, tay T 3%, tay P 25%


W0000 00:00:1794842465.983599 11582686 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842465.990818 11582687 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842465.995444 11582699 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842465.997646 11582699 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


... (81 outputs left out)

W0000 00:00:1794842692.629864 11587128 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842692.636192 11587128 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842692.640823 11587138 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842692.642903 11587138 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[44/45] OK  xin chào/13: 22 frame, pose 100%, tay T 0%, tay P 99%


W0000 00:00:1794842693.726905 11587171 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842693.733110 11587171 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842693.737375 11587182 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1794842693.739684 11587182 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[45/45] OK  xin chào/14: 21 frame, pose 100%, tay T 0%, tay P 96%
Hoàn tất: 45 thành công, 0 lỗi


## Kiểm tra dataset


In [4]:
handler = DataHandler(processor=process_sequence)
print(f'{handler.validate_dataset()} mẫu đọc được qua DataHandler')
for name, count in handler.get_class_counts().items():
    print(f'  {name}: {count}')
if handler.incomplete:
    print('Mẫu chưa hoàn chỉnh:')
    for path, reason in handler.incomplete:
        print(f'  {path}: {reason}')
print('Huấn luyện lại bằng notebooks/train.ipynb để checkpoint dùng đặc trưng mới.')

45 mẫu đọc được qua DataHandler
  cảm ơn: 15
  ten_la: 15
  xin chào: 15
Huấn luyện lại bằng notebooks/train.ipynb để checkpoint dùng đặc trưng mới.
